# 03. Entrenar modelos y generar predicciones

Este notebook ejecuta el ajuste supervisado local de T5S, BARTO, mBART-50 y uMT5-Base. Usa solamente los CSV congelados creados por el notebook 02. No emplea APIs de generación ni scripts externos.

**Importante:** `test.csv` no participa en el entrenamiento ni en la selección del checkpoint; solo se usa para generar las predicciones que evaluará el notebook 04.

## Requisitos

Con el entorno virtual activo, instala una vez las dependencias: `pip install -e '.[analysis,models,dev]'`. Los pesos base se descargarán desde Hugging Face la primera vez y se guardarán en su caché local.

In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file() and (ROOT.parent / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise RuntimeError('Abre el notebook desde la raíz del repositorio o desde notebooks/.')

FINAL_DIR = ROOT / 'artifacts' / 'sem1' / 'final'
RUN_NAME = 'current'  # nombre estable; cambia solo para iniciar otra corrida independiente
RUN_DIR = ROOT / 'artifacts' / 'sem1' / 'runs' / RUN_NAME
SPLIT_PATHS = {name: FINAL_DIR / f'{name}.csv' for name in ('train', 'validation', 'test')}
MODEL_IDS = ['t5s_spanish', 'barto_spanish', 'mbart50_es', 'umt5_base_es']
MODELS_TO_RUN = MODEL_IDS.copy()  # puedes dejar aquí un solo modelo para una prueba local
BATCH_SIZE = 1
SEED = 20261103
CACHE_DIR = None  # por ejemplo: Path('/ruta/a/cache_huggingface')
RESUME = False  # True permite reutilizar checkpoints y predicciones ya completos

print({'final_dir': str(FINAL_DIR), 'run_dir': str(RUN_DIR), 'models': MODELS_TO_RUN})

In [ ]:
import json
import platform

from claim_extraction.registry import get_model
from claim_extraction.workflow import (
    available_device,
    generate_predictions,
    train_seq2seq_model,
    utc_now,
    validate_sem1_splits,
)

if not set(MODELS_TO_RUN).issubset(MODEL_IDS):
    raise ValueError('MODELS_TO_RUN debe contener IDs definidos en MODEL_IDS.')
if len(MODELS_TO_RUN) != len(set(MODELS_TO_RUN)):
    raise ValueError('Cada modelo debe aparecer una sola vez.')
if BATCH_SIZE < 1:
    raise ValueError('BATCH_SIZE debe ser positivo.')

splits = validate_sem1_splits(SPLIT_PATHS)
print({name: len(frame) for name, frame in splits.items()})
print({'device': available_device(), 'seed': SEED})

## Crear o retomar la corrida

Una carpeta de corrida conserva checkpoints, predicciones y un manifiesto local. Con `RESUME = False`, el notebook se niega a mezclar una corrida anterior con una nueva.

In [ ]:
# El manifiesto deja registrados los parámetros de la corrida y evita mezclar configuraciones.
RUN_MANIFEST = RUN_DIR / 'run_manifest.json'
STATUS_PATH = RUN_DIR / 'run_status.json'

if RUN_DIR.exists() and any(RUN_DIR.iterdir()) and not RESUME:
    raise FileExistsError(
        f'La corrida ya existe en {RUN_DIR}. Activa RESUME o cambia RUN_NAME; no se sobrescribe.'
    )

for directory in (RUN_DIR / 'models', RUN_DIR / 'predictions'):
    directory.mkdir(parents=True, exist_ok=True)

if RUN_MANIFEST.is_file():
    run_manifest = json.loads(RUN_MANIFEST.read_text(encoding='utf-8'))
else:
    run_manifest = {
        'run_name': RUN_NAME,
        'started_at_utc': utc_now(),
        'device': available_device(),
        'platform': platform.platform(),
        'seed': SEED,
        'split_rows': {name: len(frame) for name, frame in splits.items()},
        'models': {model_id: {
            'display_name': get_model(model_id).display_name,
            'base_checkpoint': get_model(model_id).base_checkpoint,
            'training': get_model(model_id).training,
            'generation': get_model(model_id).generation,
        } for model_id in MODELS_TO_RUN},
    }
    RUN_MANIFEST.write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2), encoding='utf-8')

status = {'status': 'running', 'updated_at_utc': utc_now(), 'models': {}}
STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
print(f'Corrida preparada: {RUN_DIR.relative_to(ROOT)}')

## Ejecutar entrenamiento e inferencia

Esta celda puede tardar. Cada modelo se ajusta con `train.csv`, selecciona su checkpoint por pérdida de validación y luego genera un CSV de predicciones sobre los 40 casos de prueba. Si una ejecución se interrumpe, activa `RESUME` para reutilizar los componentes ya terminados.

In [ ]:
# Todos los modelos se entrenan con el mismo train/validation y predicen el mismo test fijo.
for model_id in MODELS_TO_RUN:
    model_dir = RUN_DIR / 'models' / model_id
    checkpoint = model_dir / 'best'
    prediction_path = RUN_DIR / 'predictions' / f'{model_id}.csv'
    try:
        if checkpoint.is_dir() and RESUME:
            print(f'{model_id}: checkpoint existente; se reutiliza.')
        else:
            status['models'][model_id] = {'status': 'training'}
            STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
            manifest = train_seq2seq_model(
                model_id=model_id,
                train=splits['train'],
                validation=splits['validation'],
                output_dir=model_dir,
                seed=SEED,
                batch_size=BATCH_SIZE,
                cache_dir=CACHE_DIR,
            )
            print(f"{model_id}: entrenamiento terminado en {manifest['device']}.")

        if prediction_path.is_file() and RESUME:
            print(f'{model_id}: predicciones existentes; se reutilizan.')
        else:
            status['models'][model_id] = {'status': 'generating', 'checkpoint': str(checkpoint)}
            STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
            generated = generate_predictions(
                model_id=model_id,
                checkpoint=checkpoint,
                candidates=splits['test'],
                output_path=prediction_path,
                batch_size=BATCH_SIZE,
                cache_dir=CACHE_DIR,
            )
            if len(generated) != len(splits['test']):
                raise AssertionError(f'{model_id} no generó las 40 predicciones esperadas.')

        status['models'][model_id] = {
            'status': 'ready_for_evaluation',
            'checkpoint': str(checkpoint),
            'predictions': str(prediction_path),
        }
        status['updated_at_utc'] = utc_now()
        STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
    except Exception as exc:
        status['models'][model_id] = {'status': 'failed', 'error': str(exc)}
        status['status'] = 'failed'
        status['updated_at_utc'] = utc_now()
        STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
        raise

status['status'] = 'ready_for_evaluation'
status['updated_at_utc'] = utc_now()
STATUS_PATH.write_text(json.dumps(status, ensure_ascii=False, indent=2), encoding='utf-8')
print('Listo. Abre el notebook 04 para calcular las métricas sobre test.csv.')

In [ ]:
import pandas as pd

pd.DataFrame([
    {'modelo': model_id, **details}
    for model_id, details in status['models'].items()
])